In [5]:
import pandas as pd
import glob
import geopandas as gpd
from shapely import wkt

data_path1 = r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260330_공원도시과\02_집계결과\\"
data_path2 = r"D:\02_사업관리\적재확인\01_생활인구\\"
output_path = r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260330_공원도시과\\"

df1 = pd.read_csv(data_path1 + "월별 격자별 시간대 생활인구 현황.csv", sep="|", dtype={'id': str}, encoding="utf-8")
values = []
data_lists = glob.glob(data_path2 + "*_tme_p_2026*.csv", recursive=True)
for data in data_lists:
    df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str, 'id': str}, encoding="utf-8")
    values.append(df[df['hcode'].str.startswith('26530')])
    df2 = pd.concat(values, ignore_index=True)

prj_grid50 = pd.read_csv(r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260330_공원도시과\\prjarea_grid50.csv", sep="|",
                           dtype={'id': str}, encoding="utf-8")
prj_grid50 = gpd.GeoDataFrame(prj_grid50, geometry=wkt.loads(prj_grid50['geometry']), crs=5179)
grid100 = gpd.read_file(r"D:\00_자료검토\03_공간자료\griddata\\grid_마라_100M.shp", encoding="utf-8")
prj_grid50['x'] = prj_grid50.geometry.centroid.x.round(3)
prj_grid50['y'] = prj_grid50.geometry.centroid.y.round(3)
prj_grid50_pt = gpd.GeoDataFrame(prj_grid50, geometry=gpd.points_from_xy(prj_grid50['x'], prj_grid50['y']), crs=5179)
prj_g100to50 = gpd.sjoin(grid100, prj_grid50_pt, how="inner", predicate="intersects")
prj_g100to50 = prj_g100to50[['GRID_100M_','id','geometry']]
prj_g100to50.rename(columns={'GRID_100M_': 'gid'}, inplace=True)

cols = [col for col in df2.columns if '_t_' in col]
df2[cols] = df2[cols].astype(float)
df2['std_ymd'] = df2['std_ymd'].astype('datetime64[ns]')
df2['std_ym'] = df2['std_ymd'].astype(str).str[:7]

time_dict = dict((f'{h:02d}H', [f'h_t_{h:02d}', f'w_t_{h:02d}', f'v_t_{h:02d}']) for h in range(9, 24))
for key, value in time_dict.items():
    df2[key] = df2.loc[:, value].sum(axis=1) / 3
    
keys = [key for key in time_dict.keys()]
df2_group = df2.groupby(['std_ym','id'], as_index=False)[keys].sum().round().sort_values('std_ym')
results = pd.concat([df1, df2_group], ignore_index=True)
results['means'] = (results.loc[:, '09H':'23H'].sum(axis=1) / 14).round()
results = pd.merge(prj_g100to50, results, on="id", how="left").dropna()
results = results[['std_ym','gid','09H','10H','11H','12H','13H','14H','15H','16H','17H','18H','19H','20H','21H','22H','23H','means','geometry']]
results = gpd.GeoDataFrame(results, geometry='geometry', crs=5179)
# results.to_file(output_path + "(2026-010) 데이터 분석 컨설팅 데이터.gpkg", layer="mon_g100m_times_pop_status", driver="GPKG")

group = results.groupby('std_ym', as_index=False)[keys].sum().round().sort_values('std_ym')
melted = pd.melt(group, id_vars=['std_ym'], value_vars=keys, var_name='times', value_name='pop')
melted.to_csv(output_path + "월별 시간대 이용자 현황.csv", sep="|", encoding="utf-8", index=False)